# Cross-Domain Plant ID, Baseline 2: Plant-Pretrained DINOv2 as Feature Extractor

This notebook implements the second required baseline: **Leveraging a Plant‑Pretrained Model (DINOv2) for Cross‑Domain Plant Identification**. We use the plant‑pretrained DINOv2 model as a **frozen feature extractor**, then train a traditional classifier on top of the embeddings.

**What you'll get:**
1. Reproducible data loading using your pre-processed CSVs (`train_v2.csv`, `val_v2.csv`, `test.csv`, `test_with_groundtruth.csv`, `species_mapping.csv`).
2. DINOv2 plant-pretrained model downloaded from *Kaggle Models* using the `kagglehub` package.
3. Batch feature extraction (no fine-tuning) and caching to NumPy.
4. A multinomial Logistic Regression classifier trained on embeddings.
5. Evaluation: Top‑1 and Top‑5 accuracy on validation and test sets.
6. Exported predictions file for report/UI integration.

> Tip: If you are not running in Colab, ensure you have a GPU and sufficient RAM.

##Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip -q install kagglehub timm scikit-learn pandas numpy pillow
!apt -y install git git-lfs >/dev/null
!git lfs install >/dev/null

import os, sys, glob, json, csv, time, math, pathlib, random
from collections import OrderedDict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import kagglehub
import timm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import top_k_accuracy_score, accuracy_score

import argparse  # needed for the allowlist below
from collections import OrderedDict

##1. Github repo clone

In [ ]:
# # Fix abs_path in processed CSVs to point to Colab’s dataset location
# def remap_abs_paths(csv_path, anchor="AML_project_herbarium_dataset"):
#     df = pd.read_csv(csv_path)
#     if "abs_path" in df.columns:
#         def fix(p):
#             s = str(p).replace("\\", "/")
#             i = s.lower().find(anchor.lower())
#             if i == -1:
#                 # fallback: join rel_path if available
#                 if "rel_path" in df.columns:
#                     rel = str(df.loc[df["abs_path"] == p, "rel_path"].values[0])
#                     return os.path.join(DATA_ROOT, rel).replace("\\", "/")
#                 return s
#             rel = s[i + len(anchor) + 1 :]
#             return os.path.join(DATA_ROOT, rel).replace("\\", "/")
#         df["abs_path"] = df["abs_path"].apply(fix)
#         df.to_csv(csv_path, index=False)
#         print("Rewrote abs_path in:", csv_path)

# for csvp in [train_csv, val_csv, test_csv, test_gt_csv]:
#     remap_abs_paths(csvp)


NameError: name 'train_csv' is not defined

In [2]:
REPO_URL = "https://github.com/Brandenn28/ML.git"
CLONE_DIR = "/content/ML"
if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 "{REPO_URL}" "{CLONE_DIR}"
%cd "{CLONE_DIR}"
!git lfs pull

# Dataset paths
DATA_ROOT = "/content/ML/AML_project_herbarium_dataset"
PROCESSED_DIR = f"{DATA_ROOT}/processed"

required_files = [
    "species_mapping.csv",
    "train_v2.csv",
    "val_v2.csv",
    "test.csv",
    "test_with_groundtruth.csv",
]
missing = [f for f in required_files if not os.path.exists(os.path.join(PROCESSED_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing processed files: {missing} in {PROCESSED_DIR}")

print("DATA_ROOT:", DATA_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Processed files sample:", os.listdir(PROCESSED_DIR)[:10])

Cloning into '/content/ML'...
remote: Enumerating objects: 5132, done.
remote: Counting objects: 100% (5132/5132), done.
remote: Compressing objects: 100% (5125/5125), done.
remote: Total 5132 (delta 7), reused 5128 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (5132/5132), 576.58 MiB | 33.02 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (4973/4973), done.
/content/ML
DATA_ROOT: /content/ML/AML_project_herbarium_dataset
Processed dir: /content/ML/AML_project_herbarium_dataset/processed
Processed files sample: ['image_sizes.csv', 'train_sample_weights.csv', 'val.csv', 'class_weights.csv', 'test_with_groundtruth.csv', 'train_v2.csv', 'train.csv', 'species_with_pairs.csv', 'val_v2.csv', 'class_counts.csv']


##2. Dataset and transforms

In [3]:
# Hyperparameter setup
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
NUM_WORKERS = 2
print(f"Running on device: {device}")

# Transformation details for the images for train and evaluation
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Replace your CSVImageDataset with this
class CSVImageDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)

        rel_list = self.df["rel_path"].astype(str).tolist() if "rel_path" in self.df.columns else []
        rel_joined = [os.path.join(DATA_ROOT, p).replace("\\", "/") for p in rel_list]

        if "abs_path" in self.df.columns:
            abs_list = self.df["abs_path"].astype(str).tolist()
        else:
            abs_list = ["" for _ in rel_list]

        paths = []
        for a, r in zip(abs_list, rel_joined):
            a_norm = a.replace("\\", "/")
            p = a_norm if a_norm and os.path.exists(a_norm) else r
            paths.append(p)
        self.paths = paths

        self.labels = self.df["label_idx"].astype(int).tolist() if "label_idx" in self.df.columns else None
        self.domain = self.df["domain"].astype(str).tolist() if "domain" in self.df.columns else None
        self.transform = transform

    def __len__(self):
      return len(self.paths)

    def __getitem__(self, i):
        path = self.paths[i]
        img = Image.open(path).convert("RGB")
        x = self.transform(img) if self.transform else img
        if self.labels is not None and self.domain is not None:
            return x, self.labels[i], self.domain[i], path
        elif self.labels is not None:
            return x, self.labels[i], path
            # include getting the domain type
        elif self.domain is not None:
            return x, self.domain[i], path
        else:
            return x, path


Running on device: cuda


In [4]:
# --- GLOBAL PATH DEFINITIONS ---
# Define these once at the top so they are available everywhere
train_csv   = os.path.join(PROCESSED_DIR, "train_v2.csv")
val_csv     = os.path.join(PROCESSED_DIR, "val_v2.csv")
test_csv    = os.path.join(PROCESSED_DIR, "test.csv")
test_gt_csv = os.path.join(PROCESSED_DIR, "test_with_groundtruth.csv")

# Verify they exist
for p in [train_csv, val_csv, test_csv, test_gt_csv]:
    if not os.path.exists(p):
        print(f"WARNING: Required file not found: {p}")
# -------------------------------

In [19]:
# missing = [p for p in train_ds.paths[:200] if not os.path.exists(p)]
# print("Sample missing files in first 200:", len(missing))
# if missing:
#     print(missing[:5])

NameError: name 'train_ds' is not defined

##3. Download and load plant-pretrained DINOv2 from Kaggle Models

In [5]:
MODEL_ID = "juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/PyTorch/default"

print("Downloading Kaggle model...")
model_dir = kagglehub.model_download(MODEL_ID)
print("Model directory:", model_dir)
print("Contents:", os.listdir(model_dir))

# 1) find checkpoint including .pth.tar
ckpt_candidates = []
for pat in ("*.pt", "*.pth", "*.bin", "*.safetensors", "*.pth.tar"):
    ckpt_candidates.extend(glob.glob(os.path.join(model_dir, "**", pat), recursive=True))
if not ckpt_candidates:
    raise FileNotFoundError("No checkpoint file found in the Kaggle model folder.")
ckpt_candidates.sort()
ckpt_path = ckpt_candidates[0]
print("Checkpoint chosen:", ckpt_path)

# 2) safe load to sd then unwrap to state dict
try:
    torch.serialization.add_safe_globals([argparse.Namespace])
except Exception:
    pass

try:
    sd = torch.load(ckpt_path, map_location="cpu")
except Exception as e:
    print("Safe load failed, retrying with weights_only=False:", e)
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)

for k in ["state_dict", "model", "net", "params"]:
    if isinstance(sd, dict) and k in sd:
        sd = sd[k]
if not isinstance(sd, dict):
    raise TypeError(f"Unexpected checkpoint format: {type(sd)}")

# 3) strip common prefixes into clean_sd
clean_sd = OrderedDict()
for k, v in sd.items():
    nk = k.replace("module.", "").replace("backbone.", "").replace("model.", "")
    clean_sd[nk] = v

# 4) choose the correct DINOv2 size from embed dim
if "cls_token" not in clean_sd:
    raise KeyError("cls_token not found in checkpoint keys. Print some keys to inspect the file.")
embed_dim = clean_sd["cls_token"].shape[-1]  # 768 base, 1024 large
if embed_dim == 768:
    model_name = "vit_base_patch14_dinov2"
elif embed_dim == 1024:
    model_name = "vit_large_patch14_dinov2.lvd142m"
else:
    raise ValueError(f"Unexpected embed dim {embed_dim}")

backbone = timm.create_model(
    model_name,
    pretrained=False,
    num_classes=0,
    img_size=IMG_SIZE,
    dynamic_img_size=False,
)
print("Backbone created:", model_name)

# --- robust pos_embed handling: infer extra tokens on both sides and resize ---

def infer_extra_tokens_and_grid_len(total_tokens):
    # return (extra_tokens, grid_len) such that (total_tokens - extra_tokens) == grid_len**2
    for e in (0, 1, 2, 5):
        n = total_tokens - e
        r = int(round(n ** 0.5))
        if r * r == n:
            return e, r
    # fallback: assume no extras
    r = int(round(total_tokens ** 0.5))
    return 0, r

load_sd = clean_sd.copy()

if "pos_embed" in load_sd and hasattr(backbone, "pos_embed"):
    pe_sd = load_sd["pos_embed"]                         # [1, T_sd, C]
    pe_m  = backbone.pos_embed
    if isinstance(pe_m, torch.nn.Parameter):
        pe_m = pe_m.data                                 # [1, T_m,  C]

    T_sd = pe_sd.shape[1]
    T_m  = pe_m.shape[1]

    # infer extra tokens separately for checkpoint and model
    e_sd, gs_sd = infer_extra_tokens_and_grid_len(T_sd)
    e_m,  gs_m  = infer_extra_tokens_and_grid_len(T_m)

    # split into extras + grid for both
    extra_sd = pe_sd[:, :e_sd] if e_sd > 0 else pe_sd[:, :0]   # possibly empty
    grid_sd  = pe_sd[:, e_sd:]
    extra_m  = pe_m[:, :e_m] if e_m > 0 else pe_m[:, :0]
    grid_m   = pe_m[:, e_m:]

    # reshape checkpoint grid to [1, C, H, W], resize to model grid, reshape back
    C = pe_sd.shape[-1]
    grid_sd = grid_sd.reshape(1, gs_sd, gs_sd, C).permute(0, 3, 1, 2)  # [1,C,H,W]
    grid_sd = torch.nn.functional.interpolate(grid_sd, size=(gs_m, gs_m), mode="bicubic", align_corners=False)
    grid_sd = grid_sd.permute(0, 2, 3, 1).reshape(1, gs_m * gs_m, C)

    # reassemble with the model's expected number of extra tokens
    load_sd["pos_embed"] = torch.cat([extra_m, grid_sd], dim=1)

# drop reg tokens if the timm backbone does not define them
for k in list(load_sd.keys()):
    if k.startswith("reg_token"):
        load_sd.pop(k)

missing, unexpected = backbone.load_state_dict(load_sd, strict=False)
print(f"Loaded with missing={len(missing)}, unexpected={len(unexpected)}")


Model directory: /kaggle/input/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3
Contents: ['model_best.pth.tar', 'args.yaml', 'summary.csv']
Checkpoint chosen: /kaggle/input/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar
Backbone created: vit_base_patch14_dinov2
Loaded with missing=0, unexpected=2


##4. Extract and cache embeddings

In [6]:
## 4. Extract and cache embeddings (with Caching)

# Define cache file paths
CACHE_DIR = PROCESSED_DIR  # Saves to your existing processed folder
train_X_path = os.path.join(CACHE_DIR, "train_X.npy")
train_y_path = os.path.join(CACHE_DIR, "train_y.npy")
val_X_path   = os.path.join(CACHE_DIR, "val_X.npy")
val_y_path   = os.path.join(CACHE_DIR, "val_y.npy")

domain_map = {"field": 0, "herbarium": 1}

# Check if all cache files exist
if os.path.exists(train_X_path) and os.path.exists(train_y_path) and \
   os.path.exists(val_X_path) and os.path.exists(val_y_path):

    print("Loading cached features from disk...")
    train_X = np.load(train_X_path)
    train_y = np.load(train_y_path)
    val_X   = np.load(val_X_path)
    val_y   = np.load(val_y_path)
    print("Features loaded! (Skipped DINOv2 extraction)")

else:
    print("Cache not found. Running DINOv2 feature extraction...")
    # --- ONLY RUNS IF CACHE IS MISSING ---

    # make sure the backbone exists
    assert "backbone" in globals(), "Backbone is not defined. Run the Kaggle model load cell first."
    for p in backbone.parameters(): p.requires_grad = False
    feature_extractor = backbone.to(device).eval()

    # Setup Loaders
    train_ds = CSVImageDataset(os.path.join(PROCESSED_DIR, "train_v2.csv"), transform=train_tf)
    val_ds   = CSVImageDataset(os.path.join(PROCESSED_DIR, "val_v2.csv"),   transform=eval_tf)

    # Re-use your collate_fn
    def collate_fn(batch):
        xs, ys, domain, paths = [], [], [], []
        for item in batch:
            if len(item) == 4:
                x, y, d, p = item
                ys.append(y)
                domain.append(d)
            else:
                x, p = item
            xs.append(x)
            paths.append(p)
        xs = torch.stack(xs, 0)
        ys = torch.tensor(ys) if ys else None
        return xs, ys, domain, paths

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True)

    @torch.no_grad()
    def embed_loader(loader):
      # Additional extract, domain
        feats, labels, domain = [], [], []
        print(f"Extracting features for {len(loader.dataset)} images...")
        for i, (xb, yb, domainb,_) in enumerate(loader):
            xb = xb.to(device)
            f = feature_extractor(xb).detach().cpu()
            feats.append(f)
            if yb is not None: labels.append(yb)
            if domainb is not None:
              mapped = [domain_map[d] for d in domainb]
              domain.extend(mapped)
            if (i+1) % 10 == 0: print(f"Processed batch {i+1}/{len(loader)}")
        return torch.cat(feats, 0).numpy(), torch.cat(labels, 0).numpy(), np.array(domain)

    print("Extracting Train...")
    train_X, train_y, train_domain = embed_loader(train_loader)
    print("Extracting Val...")
    val_X, val_y, val_domain = embed_loader(val_loader)

    # Save to disk for next time
    np.save(train_X_path, train_X)
    np.save(train_y_path, train_y)
    np.save(os.path.join(CACHE_DIR, "train_domain.npy"), train_domain)
    np.save(val_X_path, val_X)
    np.save(val_y_path, val_y)
    np.save(os.path.join(CACHE_DIR, "val_domain.npy"), val_domain)
    print(f"Features cached to {CACHE_DIR}")


Cache not found. Running DINOv2 feature extraction...
Extracting Train...
Extracting features for 3796 images...
Processed batch 10/60
Processed batch 20/60
Processed batch 30/60
Processed batch 40/60
Processed batch 50/60
Processed batch 60/60
Extracting Val...
Extracting features for 948 images...
Processed batch 10/15
Features cached to /content/ML/AML_project_herbarium_dataset/processed


In [8]:
# train_y_path = os.path.join(CACHE_DIR, "train_y.npy")
# train_y_path = os.path.join(CACHE_DIR, "train_y.npy")
# val_y_path = os.path.join(CACHE_DIR, "val_y.npy")
# val_x_path = os.path.join(CACHE_DIR, "val_X.npy")

# train_X = torch.tensor(train_X, dtype=torch.float32)
# train_y = torch.tensor(train_y, dtype=torch.long)
# val_X   = torch.tensor(val_X, dtype=torch.float32)
# val_y   = torch.tensor(val_y, dtype=torch.long)

In [21]:
class ResidualAdapter(nn.Module):
    def __init__(self, input_dim, bottleneck=256, dropout=0.5):
        super().__init__()
        # reduces dimension to 256 and add relu + dropout to introduce
        # nonlinearity + regularization
        self.down = nn.Linear(input_dim, bottleneck)
        self.act = nn.ReLU(inplace=True)
        # Reproject back up to 768 size
        self.up = nn.Linear(bottleneck, input_dim)
        self.dropout = nn.Dropout(dropout)
        # small init
        nn.init.normal_(self.up.weight, std=1e-3)
        nn.init.normal_(self.down.weight, std=1e-2)

    def forward(self, x):
        delta = self.up(self.act(self.down(x)))
        delta = self.dropout(delta)
        return x + delta  # residual skip connection

In [23]:
# Cosine similarity score instead of weights
class CosineClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, scale=10.0):
        super().__init__()
        self.W = nn.Parameter(torch.randn(num_classes, input_dim))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
        self.scale = scale

    def forward(self, x):
# Normalist embeddings and help rare class generalize
        x_norm = F.normalize(x, dim=1)
        W_norm = F.normalize(self.W, dim=1)
        logits = self.scale * x_norm @ W_norm.t()
        return logits

In [24]:
def rbf_mmd_loss(x, y, gamma=None):
    if x.numel() == 0 or y.numel() == 0:
        return x.new_tensor(0.0)

    # median heuristic for gamma
    if gamma is None:
        with torch.no_grad():
            z = torch.cat([x, y], dim=0)
            n = z.shape[0]
            if n > 1024:
                idx = torch.randperm(n)[:1024]
                z = z[idx]
            dists = torch.cdist(z, z, p=2)
            median = torch.median(dists)
            gamma = 1.0 / (2 * (median ** 2 + 1e-12))

    K_xx = torch.exp(-gamma * torch.cdist(x, x, p=2) ** 2)
    K_yy = torch.exp(-gamma * torch.cdist(y, y, p=2) ** 2)
    K_xy = torch.exp(-gamma * torch.cdist(x, y, p=2) ** 2)

    mmd = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()
    return mmd

In [26]:
class EmbeddingDataset(Dataset):
    def __init__(self, features, labels, domain):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.domain = torch.tensor(domain, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], self.domain[idx]

# -----------------------------
# 5. Load cached embeddings
# -----------------------------
train_X = np.load(os.path.join(CACHE_DIR,"train_X.npy"))
train_y = np.load(os.path.join(CACHE_DIR,"train_y.npy"))
train_domain = np.load(os.path.join(CACHE_DIR,"train_domain.npy"))

val_X = np.load(os.path.join(CACHE_DIR,"val_X.npy"))
val_y = np.load(os.path.join(CACHE_DIR,"val_y.npy"))
val_domain = np.load(os.path.join(CACHE_DIR, "val_domain.npy"))

train_ds = EmbeddingDataset(train_X, train_y, train_domain)
val_ds = EmbeddingDataset(val_X, val_y, val_domain)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# 6. Model
# -----------------------------
INPUT_DIM = train_X.shape[1]
NUM_CLASSES = len(np.unique(train_y))

adapter = ResidualAdapter(INPUT_DIM).to(device)
classifier = CosineClassifier(INPUT_DIM, NUM_CLASSES).to(device)

params = list(adapter.parameters()) + list(classifier.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)

# -----------------------------
# 7. Training loop
# -----------------------------
NUM_EPOCHS = 50
lambda_mmd = 0.15

for epoch in range(NUM_EPOCHS):
    adapter.train()
    classifier.train()
    total_loss, total_cls, total_mmd = 0.0, 0.0, 0.0

    for features, labels, domains in train_loader:
        features, labels, domains = features.to(device), labels.to(device), domains.to(device)

        optimizer.zero_grad()
        adapted = adapter(features)
        logits = classifier(adapted)

        ce_loss = F.cross_entropy(logits, labels)

        # MMD between herbarium (1) and field (0)
        herb_feats = adapted[domains==1]
        field_feats = adapted[domains==0]
        mmd = rbf_mmd_loss(field_feats, herb_feats)

        loss = ce_loss + lambda_mmd * mmd
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * features.size(0)
        total_cls += ce_loss.item() * features.size(0)
        total_mmd += mmd.item() * features.size(0)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {total_loss/len(train_ds):.4f} | "
          f"CE: {total_cls/len(train_ds):.4f} | "
          f"MMD: {total_mmd/len(train_ds):.4f}")

    # Validation
    top1, top5 = validate(adapter, classifier, val_loader, device)
    print(f"Validation Top-1: {top1:.4f} | Top-5: {top5:.4f}")


Epoch 1/50 | Loss: 3.4791 | CE: 3.4573 | MMD: 0.1449
Validation Top-1: 0.5464 | Top-5: 0.8112
Epoch 2/50 | Loss: 1.9720 | CE: 1.9547 | MMD: 0.1155
Validation Top-1: 0.6340 | Top-5: 0.8829
Epoch 3/50 | Loss: 1.4722 | CE: 1.4554 | MMD: 0.1119
Validation Top-1: 0.6857 | Top-5: 0.9008
Epoch 4/50 | Loss: 1.1861 | CE: 1.1702 | MMD: 0.1060
Validation Top-1: 0.7099 | Top-5: 0.8935
Epoch 5/50 | Loss: 0.9930 | CE: 0.9760 | MMD: 0.1127
Validation Top-1: 0.7131 | Top-5: 0.9040
Epoch 6/50 | Loss: 0.8351 | CE: 0.8185 | MMD: 0.1104
Validation Top-1: 0.7141 | Top-5: 0.9040
Epoch 7/50 | Loss: 0.7174 | CE: 0.7013 | MMD: 0.1069
Validation Top-1: 0.7215 | Top-5: 0.9040
Epoch 8/50 | Loss: 0.6249 | CE: 0.6091 | MMD: 0.1057
Validation Top-1: 0.7278 | Top-5: 0.9061
Epoch 9/50 | Loss: 0.5506 | CE: 0.5345 | MMD: 0.1073
Validation Top-1: 0.7236 | Top-5: 0.8998
Epoch 10/50 | Loss: 0.4893 | CE: 0.4729 | MMD: 0.1093
Validation Top-1: 0.7194 | Top-5: 0.8977
Epoch 11/50 | Loss: 0.4401 | CE: 0.4242 | MMD: 0.1060
Valid

In [27]:
def validate(adapter, classifier, val_loader, device):
    adapter.eval()
    classifier.eval()
    top1_correct, top5_correct, total = 0, 0, 0

    with torch.no_grad():
        for features, labels, _ in val_loader:
            features, labels = features.to(device), labels.to(device)
            adapted = adapter(features)
            logits = classifier(adapted)
            prob = torch.softmax(logits, dim=1)

            # Top-1
            pred1 = prob.argmax(dim=1)
            top1_correct += (pred1 == labels).sum().item()

            # Top-5
            top5_pred = prob.topk(5, dim=1).indices
            top5_correct += sum([labels[i] in top5_pred[i] for i in range(labels.size(0))])

            total += labels.size(0)

    top1 = top1_correct / total
    top5 = top5_correct / total
    return top1, top5

In [ ]:
# # make sure the backbone exists from the previous model cell
# assert "backbone" in globals(), "Backbone is not defined. Run the Kaggle model load cell first."

# # freeze and use the backbone directly as the feature extractor
# for p in backbone.parameters():
#     p.requires_grad = False
# feature_extractor = backbone.to(device).eval()

# print("Feature extractor ready on", device)


Feature extractor ready on cuda


In [ ]:
# # Build datasets from processed CSVs
# train_csv = f"{PROCESSED_DIR}/train_v2.csv"
# val_csv   = f"{PROCESSED_DIR}/val_v2.csv"
# test_csv  = f"{PROCESSED_DIR}/test.csv"
# test_gt_csv = f"{PROCESSED_DIR}/test_with_groundtruth.csv"

# train_ds = CSVImageDataset(train_csv, transform=train_tf)
# val_ds   = CSVImageDataset(val_csv,   transform=eval_tf)

# def collate_fn(batch):
#     xs, ys, paths = [], [], []
#     for item in batch:
#         if len(item) == 3:
#             x, y, p = item
#             ys.append(y)
#         else:
#             x, p = item
#         xs.append(x)
#         paths.append(p)
#     xs = torch.stack(xs, 0)
#     ys = torch.tensor(ys) if ys else None
#     return xs, ys, paths

# @torch.no_grad()
# def embed_loader(loader):
#     feats, labels, filepaths = [], [], []
#     for xb, yb, paths in loader:
#         xb = xb.to(device)
#         f = feature_extractor(xb).detach().cpu()
#         feats.append(f)
#         filepaths.extend(paths)
#         if yb is not None:
#             labels.append(yb)
#     feats = torch.cat(feats, 0).numpy()
#     y = torch.cat(labels, 0).numpy() if labels else None
#     return feats, y, filepaths


In [ ]:
# BATCH_SIZE = 64
# NUM_WORKERS = 0
# PERSISTENT = False
# PIN = torch.cuda.is_available()


# train_loader = DataLoader(
#     train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     collate_fn=collate_fn,
#     persistent_workers=PERSISTENT,
#     pin_memory=PIN,
# )

# val_loader = DataLoader(
#     val_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     collate_fn=collate_fn,
#     persistent_workers=PERSISTENT,
#     pin_memory=PIN,
# )


In [ ]:
# train_X, train_y, _ = embed_loader(train_loader)
# val_X,   val_y,   _ = embed_loader(val_loader)


##5. Train multinomial Logistic Regression and evaluate on val

In [ ]:
weights_csv = os.path.join(PROCESSED_DIR, "train_sample_weights.csv")
sample_weight = None
if os.path.exists(weights_csv):
    wdf = pd.read_csv(weights_csv)
    if len(wdf) == len(train_X):
        sample_weight = wdf["sample_weight"].values

clf = LogisticRegression(
    penalty="l2",
    solver="saga",
    multi_class="multinomial",
    max_iter=2000,
    n_jobs=-1,
    verbose=0
)
clf.fit(train_X, train_y, sample_weight=sample_weight)

val_prob = clf.predict_proba(val_X)
val_pred = val_prob.argmax(1)
top1 = accuracy_score(val_y, val_pred)
top5 = top_k_accuracy_score(val_y, val_prob, k=5, labels=np.arange(val_prob.shape[1]))
print(f"Validation Top-1: {top1:.4f}")
print(f"Validation Top-5: {top5:.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Validation Top-1: 0.6804
Validation Top-5: 0.8861


##6. Test inference and Top-1 Top-5

In [ ]:
## 6. Test inference and Top-1 Top-5 (Robust + Cached Version)

# Load test DataFrames
test_df = pd.read_csv(test_csv)
gt_df = pd.read_csv(test_gt_csv)

# Define cache paths for test features
test_X_path = os.path.join(CACHE_DIR, "test_X.npy")
test_paths_path = os.path.join(CACHE_DIR, "test_rel_paths.npy")

# Check if test features are already cached
if os.path.exists(test_X_path) and os.path.exists(test_paths_path):
    print("Loading cached TEST features...")
    test_X = np.load(test_X_path)
    test_rel_paths = np.load(test_paths_path, allow_pickle=True)

else:
    print("Cache miss. Running TEST feature extraction...")

    # --- Ensure feature_extractor exists globally BEFORE defining the function ---
    if 'feature_extractor' not in globals():
         print("Initializing DINOv2 backbone for testing...")
         feature_extractor = backbone.to(device).eval()

    class TestRelPathDataset(Dataset):
        def __init__(self, df, root_dir, transform):
            self.df = df
            self.root_dir = root_dir
            self.transform = transform
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            row = self.df.iloc[i]
            rel_p = row['rel_path']
            full_path = os.path.join(self.root_dir, rel_p).replace("\\", "/")
            try:
                img = Image.open(full_path).convert("RGB")
            except FileNotFoundError:
                if "abs_path" in row and os.path.exists(row["abs_path"]):
                     img = Image.open(row["abs_path"]).convert("RGB")
                else:
                     raise FileNotFoundError(f"Could not find {rel_p} at {full_path}")
            return self.transform(img), rel_p

    test_ds = TestRelPathDataset(test_df, DATA_ROOT, eval_tf)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    print(f"Running inference on {len(test_ds)} test images...")

    @torch.no_grad()
    def get_test_features(loader):
        feats, r_paths = [], []
        for x, rp in loader:
            x = x.to(device)
            # Now this safely finds the global feature_extractor
            f = feature_extractor(x).detach().cpu()
            feats.append(f)
            r_paths.extend(rp)
        return torch.cat(feats, 0).numpy(), np.array(r_paths)

    test_X, test_rel_paths = get_test_features(test_loader)

    # Save to cache
    np.save(test_X_path, test_X)
    np.save(test_paths_path, test_rel_paths)
    print("Test features cached to disk.")

# --- CONTINUE WITH PREDICTION & EVALUATION ---

# Get probabilities from the trained classifier
test_prob = clf.predict_proba(test_X)

# Create lookup: rel_path -> prediction probabilities
pred_lookup = {rp: prob for rp, prob in zip(test_rel_paths, test_prob)}

y_true, y_pred_top1, y_pred_top5_ok = [], [], []
missing_preds = 0

for _, row in gt_df.iterrows():
    rp = row['rel_path']
    true_label = int(row['label_idx'])

    if rp in pred_lookup:
        probs = pred_lookup[rp]
        pred_label = probs.argmax()
        top5_labels = np.argsort(-probs)[:5]

        y_true.append(true_label)
        y_pred_top1.append(pred_label)
        y_pred_top5_ok.append(1 if true_label in top5_labels else 0)
    else:
        missing_preds += 1

if missing_preds > 0:
    print(f"Warning: {missing_preds} images in ground truth were not found in predictions.")

print(f"Test Top-1 Accuracy: {(np.array(y_true) == np.array(y_pred_top1)).mean():.4f}")
print(f"Test Top-5 Accuracy: {np.array(y_pred_top5_ok).mean():.4f}")

# Save predictions to CSV
pd.DataFrame({
    "rel_path": test_rel_paths,
    "pred_label_idx": test_prob.argmax(1)
}).to_csv(f"{PROCESSED_DIR}/test_predictions.csv", index=False)
print("Saved test_predictions.csv")

Cache miss. Running TEST feature extraction...
Running inference on 207 test images...
Test features cached to disk.
Test Top-1 Accuracy: 0.4879
Test Top-5 Accuracy: 0.6860
Saved test_predictions.csv
